# DMD — Reproducible experiment runner

One notebook to **run a whole grid of DMD experiments and visualize them**, built for **Google Colab**.

Everything is driven by a single **configuration cell**: pick the task, the list of encoders, the list of datasets, the seeds, the number of epochs, and the shared hyperparameters — then *Runtime ▸ Run all*.

The grid is the Cartesian product **datasets × encoders × seeds**. Encoders are the comparison axis (color), datasets are facets, seeds give mean ± std.

**Outputs** (saved under `results/<RUN_NAME>/`):
1. `fig1_learning_curves.png` — test accuracy vs. epoch (mean ± std over seeds), one panel per dataset.
2. `fig2_final_accuracy_bars.png` — final test accuracy (test @ best val), grouped bars.
3. `fig3_seed_distribution.png` — per-seed spread (box + points).
4. `fig4_summary_table.png` — visual summary table (+ `summary.csv`).
5. `results.pkl` / `config.json` — raw results and the exact config used.


## 1. Install dependencies & fetch the code

In [ ]:
# Colab already ships torch; we only add PyTorch Geometric.
# PyG >= 2.3 uses torch's native scatter -> no torch-scatter/torch-sparse needed.
!pip -q install torch_geometric

# --- Fetch the repository from GitHub ---
import os, sys

REPO_URL  = 'https://github.com/AlGoRythm3000/Differentiable-Motif-Discovery.git'
REPO_BRANCH = None            # e.g. 'main' or a feature branch; None -> default branch
CLONE_DIR = '/content/Differentiable-Motif-Discovery'

if not os.path.isdir(CLONE_DIR):
    branch = f'-b {REPO_BRANCH} ' if REPO_BRANCH else ''
    !git clone {branch}{REPO_URL} {CLONE_DIR}
else:
    print('Repo already cloned, updating (git pull)...')
    !git -C {CLONE_DIR} pull --ff-only

# Root folder that contains train.py / models/ / tasks/
REPO_DIR = CLONE_DIR
assert os.path.isfile(os.path.join(REPO_DIR, 'train.py')), (
    f'train.py not found in {REPO_DIR!r}. Check REPO_URL / REPO_BRANCH.')
os.chdir(REPO_DIR)                       # so TUDataset downloads into ./datasets
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('Repo OK :', REPO_DIR)

## 2. Experiment configuration — **edit everything here**

In [ ]:
import torch

# === What to compare =======================================================
TASK     = 'graph_classification'   # 'graph_classification' | 'node_classification'
ENCODERS = ['gcn', 'gin']           # Stage-1 encoders (models/graph_embeddings.py)
DATASETS = ['MUTAG', 'PROTEINS', 'NCI1']    # graph_classification: any of NCI1 / MUTAG / PROTEINS
                                    # node_classification : use ['path_of_cliques']
SEEDS    = [0, 1, 2, 3, 4]          # one run per seed -> mean / std across seeds
EPOCHS   = 150

# === Shared hyperparameters (applied to EVERY run) =========================
# Keys are train.py argparse names. To sweep one of these, change it and
# re-run under a NEW run_name (or set FORCE = True) so the cache is not reused.
HP = dict(
    # model
    hidden_dim=64, latent_dim=32,
    motif_hidden_dim=32, motif_out_dim=32,
    top_k=4, selector_tau=0.5,
    selector_soft=False,        # True  -> soft (non straight-through) Stage-4 selection
    no_original_edges=False,    # True  -> rewired-1-skeleton-only ablation
    # optimization
    lr=0.005, weight_decay=5e-4, sparsity_weight=0.05,
    # graph_classification only
    batch_size=32, train_frac=0.6, val_frac=0.2, data_root='datasets',
    # node_classification only (ignored for graph_classification)
    num_cliques=8, clique_size=6, feature_dim=16,
)

# === Bookkeeping ===========================================================
RUN_NAME = 'encoders_gcn_vs_gin'    # results saved under results/<RUN_NAME>/
FORCE    = False                    # True -> recompute even if a cached result exists
DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

OUT_DIR = os.path.join(REPO_DIR, 'results', RUN_NAME)
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS_PKL = os.path.join(OUT_DIR, 'results.pkl')

n_runs = len(DATASETS) * len(ENCODERS) * len(SEEDS)
print(f'Task    : {TASK}')
print(f'Device  : {DEVICE}')
print(f'Grid    : {len(DATASETS)} datasets x {len(ENCODERS)} encoders x {len(SEEDS)} seeds'
      f' = {n_runs} runs of {EPOCHS} epochs')
print(f'Output  : {OUT_DIR}')

## 3. Training loop (faithful to `train.py`, device-aware, records per-epoch history)

In [ ]:
import time

import utils
from train import build_arg_parser, TASKS
from tasks import node_classification as nc
from models.dmd_model import DMDModel
from tools.losses import DMDLoss
from tools.metrics import accuracy


def make_args(**overrides):
    """Start from train.py defaults, then override any argparse field by name."""
    args = build_arg_parser().parse_args([])
    for k, v in overrides.items():
        if not hasattr(args, k):
            raise KeyError(f'Unknown argument {k!r} (see train.py build_arg_parser).')
        setattr(args, k, v)
    return args


def _graph_epoch(model, loader, criterion, device, optimizer=None):
    """One pass over a DataLoader (train if optimizer given). Mirrors
    tasks/graph_classification._run_loader, plus .to(device) for GPU."""
    is_train = optimizer is not None
    model.train(is_train)
    tot_loss = tot_correct = 0.0
    n_graphs = 0
    for batch in loader:
        batch = batch.to(device)
        if is_train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(is_train):
            logits, structure = model(batch.x, batch.edge_index, batch=batch.batch)
            mask = torch.ones(logits.size(0), dtype=torch.bool, device=device)
            loss_out = criterion(logits, batch.y, mask, structure)
        if is_train:
            loss_out.total.backward()
            optimizer.step()
        bs = batch.num_graphs
        tot_loss    += loss_out.total.item() * bs
        tot_correct += accuracy(logits, batch.y, mask) * bs
        n_graphs    += bs
    return {'loss': tot_loss / n_graphs, 'accuracy': tot_correct / n_graphs}


def run_experiment(dataset, encoder, seed, task, epochs, hp, device, verbose=False):
    """Train one DMD model; return per-epoch history + test acc @ best val."""
    args = make_args(
        task=task,
        dataset=(None if task == 'node_classification' else dataset),
        encoder=encoder, seed=seed, epochs=epochs, results_dir='', **hp,
    )
    utils.set_seed(args.seed)                    # before load_dataset (split depends on seed)
    data, input_dim, num_classes = TASKS[task].load_dataset(args)

    model = DMDModel(
        input_dim=input_dim, hidden_dim=args.hidden_dim, latent_dim=args.latent_dim,
        motif_hidden_dim=args.motif_hidden_dim, motif_out_dim=args.motif_out_dim,
        num_classes=num_classes, encoder_type=args.encoder, top_k=args.top_k,
        selector_tau=args.selector_tau, selector_hard=not args.selector_soft,
        include_original_edges=not args.no_original_edges,
    ).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    criterion = DMDLoss(sparsity_weight=args.sparsity_weight)

    is_graph = (task == 'graph_classification')
    if not is_graph:
        data = data.to(device)                   # single-graph node task

    hist = {'train_loss': [], 'train_acc': [], 'val_acc': [], 'test_acc': []}
    best_val, test_at_best = 0.0, 0.0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        if is_graph:
            tr = _graph_epoch(model, data.train_loader, criterion, device, optimizer=optimizer)
            va = _graph_epoch(model, data.val_mask,  criterion, device)   # val_mask  = val_loader
            te = _graph_epoch(model, data.test_mask, criterion, device)   # test_mask = test_loader
        else:
            tr = nc.train_step(model, data, optimizer, criterion)
            va = nc.eval_step(model, data, criterion, data.val_mask)
            te = nc.eval_step(model, data, criterion, data.test_mask)
        if va['accuracy'] > best_val:
            best_val, test_at_best = va['accuracy'], te['accuracy']
        hist['train_loss'].append(tr['loss'])
        hist['train_acc'].append(tr['accuracy'])
        hist['val_acc'].append(va['accuracy'])
        hist['test_acc'].append(te['accuracy'])
        if verbose and (epoch % 30 == 0 or epoch == 1):
            print(f'    ep {epoch:03d} | loss {tr["loss"]:.3f} | val {va["accuracy"]:.3f} '
                  f'| test {te["accuracy"]:.3f}')
    return {'history': hist, 'best_val_acc': best_val,
            'test_acc_at_best_val': test_at_best, 'seconds': time.time() - t0}

## 4. Run the whole grid

Results are cached to `results.pkl`; re-running only computes what is missing. Set `FORCE = True` (config cell) to recompute everything. Changing `HP` **without** changing `RUN_NAME` reuses the old cache — pick a new `RUN_NAME` for a new hyperparameter setting.

In [ ]:
import pickle, json, itertools

# Snapshot the exact config next to the results (reproducibility).
with open(os.path.join(OUT_DIR, 'config.json'), 'w') as f:
    json.dump({'task': TASK, 'encoders': ENCODERS, 'datasets': DATASETS,
               'seeds': SEEDS, 'epochs': EPOCHS, 'hp': HP}, f, indent=2)

results = {}
if os.path.isfile(RESULTS_PKL) and not FORCE:
    with open(RESULTS_PKL, 'rb') as f:
        results = pickle.load(f)
    print(f'{len(results)} runs loaded from cache.')

grid = list(itertools.product(DATASETS, ENCODERS, SEEDS))
for i, (dataset, encoder, seed) in enumerate(grid, start=1):
    key = (dataset, encoder, seed)
    if key in results and not FORCE:
        continue
    print(f'[{i}/{len(grid)}] {dataset} | {encoder} | seed {seed} ...', flush=True)
    res = run_experiment(dataset, encoder, seed, TASK, EPOCHS, HP, DEVICE, verbose=True)
    results[key] = res
    with open(RESULTS_PKL, 'wb') as f:      # incremental save (crash-safe)
        pickle.dump(results, f)
    print(f'    -> test @ best val = {res["test_acc_at_best_val"]:.4f} ({res["seconds"]:.0f}s)')
print('\nDone.')

## 5. Plot style & helpers (colorblind-safe palette)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Okabe-Ito categorical palette (colorblind-safe), fixed order per encoder.
PALETTE = {'gcn': '#0072B2', 'gin': '#E69F00', 'sage': '#009E73',
           'gat': '#D55E00', 'sgc': '#CC79A7', 'graphgps': '#56B4E9'}
_FALLBACK = ['#000000', '#999999', '#F0E442']
def color_of(enc):
    if enc in PALETTE:
        return PALETTE[enc]
    return _FALLBACK[hash(enc) % len(_FALLBACK)]

plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'grid.linewidth': 0.6,
    'legend.frameon': False,
})

def stacked(dataset, encoder, field):
    """[n_seeds, n_epochs] matrix of a history field (missing runs skipped)."""
    rows = [results[(dataset, encoder, s)]['history'][field]
            for s in SEEDS if (dataset, encoder, s) in results]
    return np.array(rows)

def finals(dataset, encoder):
    """Vector of test-acc @ best-val across seeds."""
    return np.array([results[(dataset, encoder, s)]['test_acc_at_best_val']
                     for s in SEEDS if (dataset, encoder, s) in results])

### Figure 1 — Learning curves (test accuracy, mean ± std over seeds)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(6.2 * len(DATASETS), 4.6), squeeze=False)
axes = axes[0]

for ax, dataset in zip(axes, DATASETS):
    for enc in ENCODERS:
        M = stacked(dataset, enc, 'test_acc')
        if M.size == 0:
            continue
        ep = np.arange(1, M.shape[1] + 1)
        mean, std = M.mean(0), M.std(0)
        c = color_of(enc)
        ax.plot(ep, mean, color=c, lw=2, label=enc.upper())
        ax.fill_between(ep, mean - std, mean + std, color=c, alpha=0.18, linewidth=0)
        ax.annotate(f'{mean[-1]:.2f}', (ep[-1], mean[-1]), color=c, fontsize=10,
                    fontweight='bold', xytext=(4, 0), textcoords='offset points', va='center')
    ax.set_title(dataset)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Test accuracy')
    ax.set_xlim(1, EPOCHS)
    ax.legend(title='Encoder')

fig.suptitle(f'Learning curves — mean ± std ({len(SEEDS)} seeds)', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig1_learning_curves.png'))
plt.show()

### Figure 2 — Final test accuracy (test @ best val), grouped bars

In [ ]:
fig, ax = plt.subplots(figsize=(1.6 * len(DATASETS) + 3, 4.6))

n_enc = len(ENCODERS)
bar_w = 0.72 / n_enc
x = np.arange(len(DATASETS))

for j, enc in enumerate(ENCODERS):
    means = np.array([finals(d, enc).mean() if finals(d, enc).size else np.nan for d in DATASETS])
    stds  = np.array([finals(d, enc).std()  if finals(d, enc).size else 0.0    for d in DATASETS])
    offset = (j - (n_enc - 1) / 2) * bar_w
    c = color_of(enc)
    bars = ax.bar(x + offset, means, bar_w * 0.92, yerr=stds, capsize=4,
                  color=c, edgecolor='white', linewidth=1.2, label=enc.upper(),
                  error_kw=dict(ecolor='#444', lw=1.2))
    for b, m in zip(bars, means):
        if np.isfinite(m):
            ax.annotate(f'{m:.2f}', (b.get_x() + b.get_width() / 2, m),
                        xytext=(0, 4), textcoords='offset points',
                        ha='center', fontsize=9.5, fontweight='bold', color='#222')

ax.set_xticks(x)
ax.set_xticklabels(DATASETS)
ax.set_ylabel('Test accuracy @ best val')
ax.set_ylim(0, 1.0)
ax.set_title(f'Final accuracy by encoder — mean ± std ({len(SEEDS)} seeds)')
ax.legend(title='Encoder')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig2_final_accuracy_bars.png'))
plt.show()

### Figure 3 — Per-seed spread (box + points)

In [ ]:
fig, axes = plt.subplots(1, len(DATASETS), figsize=(4.2 * len(DATASETS), 4.4),
                         squeeze=False, sharey=True)
axes = axes[0]
rng = np.random.RandomState(0)

for ax, dataset in zip(axes, DATASETS):
    positions = np.arange(len(ENCODERS))
    box_data = [finals(dataset, enc) for enc in ENCODERS]
    bp = ax.boxplot(box_data, positions=positions, widths=0.5,
                    patch_artist=True, showfliers=False,
                    medianprops=dict(color='#222', lw=1.5))
    for patch, enc in zip(bp['boxes'], ENCODERS):
        patch.set_facecolor(color_of(enc)); patch.set_alpha(0.25); patch.set_edgecolor(color_of(enc))
    for pos, enc in zip(positions, ENCODERS):
        vals = finals(dataset, enc)
        jitter = rng.uniform(-0.09, 0.09, size=len(vals))
        ax.scatter(pos + jitter, vals, color=color_of(enc), s=40,
                   edgecolor='white', linewidth=0.8, zorder=3)
    ax.set_xticks(positions)
    ax.set_xticklabels([e.upper() for e in ENCODERS])
    ax.set_title(dataset)
    ax.set_ylim(0, 1.0)

axes[0].set_ylabel('Test accuracy @ best val')
fig.suptitle(f'Spread across {len(SEEDS)} seeds', fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'fig3_seed_distribution.png'))
plt.show()

## 6. Summary — CSV **and** visual table image

In [ ]:
import pandas as pd

rows = []
for dataset in DATASETS:
    for enc in ENCODERS:
        v = finals(dataset, enc)
        if v.size == 0:
            continue
        rows.append({'dataset': dataset, 'encoder': enc.upper(),
                     'test_acc_mean': float(v.mean()), 'test_acc_std': float(v.std()),
                     'min': float(v.min()), 'max': float(v.max()), 'n_seeds': int(v.size)})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)

# --- Visual table image ---
def _best_enc(d):
    cand = [(enc, finals(d, enc).mean()) for enc in ENCODERS if finals(d, enc).size]
    return max(cand, key=lambda t: t[1])[0].upper() if cand else None
best = {d: _best_enc(d) for d in DATASETS}

col_labels = ['Dataset', 'Encoder', 'Test acc (mean ± std)', 'Min', 'Max', 'Seeds']
cell_text, cell_colors = [], []
for _, r in df.iterrows():
    is_best = (best[r['dataset']] == r['encoder'])
    cell_text.append([r['dataset'], r['encoder'],
                      f"{r['test_acc_mean']:.3f} ± {r['test_acc_std']:.3f}",
                      f"{r['min']:.3f}", f"{r['max']:.3f}", str(int(r['n_seeds']))])
    cell_colors.append(['#eaf6ef' if is_best else 'white'] * len(col_labels))

fig, ax = plt.subplots(figsize=(9, 0.62 * len(cell_text) + 1.3))
ax.axis('off')
tbl = ax.table(cellText=cell_text, colLabels=col_labels, cellColours=cell_colors,
               cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1, 1.6)
for j in range(len(col_labels)):                       # header styling
    c = tbl[0, j]; c.set_facecolor('#333333'); c.set_text_props(color='white', fontweight='bold')
for i, (_, r) in enumerate(df.iterrows(), start=1):    # bold the best encoder per dataset
    if best[r['dataset']] == r['encoder']:
        tbl[i, 1].set_text_props(fontweight='bold')
        tbl[i, 2].set_text_props(fontweight='bold')
ax.set_title('Summary — test accuracy @ best val   (best encoder per dataset highlighted)',
             fontweight='bold', pad=16)
fig.savefig(os.path.join(OUT_DIR, 'fig4_summary_table.png'), bbox_inches='tight')
plt.show()

print('All figures + summary.csv saved to:', OUT_DIR)
df